# 05 - Luật kết hợp
## Association Rules with FP-Growth

Mục tiêu:
- Tạo transaction từ skills/tags
- Áp dụng FP-Growth
- Tính Support, Confidence, Lift
- Xuất Top 20 rules tốt nhất

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast, re, os, joblib, warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

os.makedirs('../report/images', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')
print("Ready!")

In [ ]:
df = pd.read_csv('../data/processed/unified_data.csv')
coursera_raw = pd.read_csv('../courses_en.csv', low_memory=False)

# Merge skills
if 'skills' in coursera_raw.columns:
    coursera_mask = df['source'] == 'coursera'
    n_c = coursera_mask.sum()
    orig = coursera_raw.drop_duplicates(subset=['name'] if 'name' in coursera_raw.columns else coursera_raw.columns[1:2]).head(n_c)
    df.loc[coursera_mask, 'skills'] = orig['skills'].fillna('').values[:n_c]
else:
    df['skills'] = ''

print(f"Dataset: {df.shape}")
print(df['source'].value_counts())

## 5.1 Tạo Transactions

In [ ]:
def parse_skills(s):
    if not isinstance(s, str) or not s.strip(): return []
    try:
        items = ast.literal_eval(s)
        if isinstance(items, list):
            return [str(i).strip().lower() for i in items if str(i).strip()]
    except: pass
    return [i.strip().lower() for i in re.split(r'[,;]', s) if i.strip()]

def extract_khan_tags(title, category):
    tags = []
    if isinstance(category, str) and category.strip():
        tags.append(category.strip().lower())
    subjects = ['math', 'algebra', 'calculus', 'chemistry', 'physics', 'biology',
                'history', 'economics', 'programming', 'computer', 'english']
    title_lower = str(title).lower()
    tags.extend([s for s in subjects if s in title_lower])
    return list(set(tags))

# Build transactions
transactions = []
for _, row in df.iterrows():
    items = []
    if row.get('source') == 'coursera':
        items = parse_skills(str(row.get('skills', '')))
    else:
        items = extract_khan_tags(row.get('title',''), row.get('category',''))
    
    diff = str(row.get('difficulty','')).strip().lower()
    if diff in ['easy','medium','hard']:
        items.append(f'difficulty:{diff}')
    
    if len(items) >= 2:
        transactions.append(list(set(items)))

print(f"Total transactions: {len(transactions):,}")
print(f"Sample: {transactions[0][:5] if transactions else []}")

## 5.2 FP-Growth

In [ ]:
# Encode
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)

print(f"Transaction matrix: {te_df.shape}")
print(f"Unique items: {len(te.columns_)}")

# FP-Growth
freq_itemsets = fpgrowth(te_df, min_support=0.005, use_colnames=True)
print(f"Frequent itemsets: {len(freq_itemsets)}")
freq_itemsets.head(10)

In [ ]:
# Generate rules
rules = association_rules(freq_itemsets, metric='confidence', min_threshold=0.3)
rules = rules[rules['lift'] >= 1.0].copy()
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f"Total rules generated: {len(rules)}")
print(f"\nTop 5 rules by Lift:")
rules[['antecedents_str','consequents_str','support','confidence','lift']].head(5)

## 5.3 Top 20 Luật tốt nhất

In [ ]:
top20 = rules[['antecedents_str','consequents_str','support','confidence','lift']].head(20)
print("\n=== TOP 20 ASSOCIATION RULES (by Lift) ===")
print(top20.to_string(index=False))

# Save
rules.to_csv('../data/processed/association_rules.csv', index=False)
top20.to_csv('../data/processed/top20_rules.csv', index=False)
print("\nRules saved!")

## 5.4 Biểu đồ trực quan

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Association Rules Analysis', fontsize=16, fontweight='bold')

# Scatter Support vs Confidence (colored by Lift)
scatter = axes[0].scatter(rules['support'], rules['confidence'],
                           c=rules['lift'], cmap='YlOrRd', alpha=0.6, s=60)
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].set_title('Support vs Confidence (color=Lift)')
axes[0].set_xlabel('Support'); axes[0].set_ylabel('Confidence')

# Top 20 by Lift - bar chart
y_pos = range(len(top20))
rule_labels = [f"{r['antecedents_str'][:18]}→{r['consequents_str'][:12]}" 
               for _, r in top20.iterrows()]
axes[1].barh(list(y_pos), top20['lift'].values, color=plt.cm.RdYlGn(np.linspace(0,1,len(top20))))
axes[1].set_yticks(list(y_pos)); axes[1].set_yticklabels(rule_labels, fontsize=7)
axes[1].set_title('Top 20 Rules by Lift'); axes[1].set_xlabel('Lift')

plt.tight_layout()
plt.savefig('../report/images/nb05_association_rules.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bubble chart
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(top20['support'], top20['confidence'],
                     s=top20['lift']*300, c=top20['lift'],
                     cmap='Reds', alpha=0.7, edgecolors='darkred', lw=1)
plt.colorbar(scatter, ax=ax, label='Lift')
for _, row in top20.head(10).iterrows():
    ax.annotate(f"{row['antecedents_str'][:15]}→{row['consequents_str'][:10]}",
                (row['support'], row['confidence']), fontsize=7, ha='center', va='bottom')
ax.set_title('Association Rules Bubble Chart (size=Lift)', fontsize=14, fontweight='bold')
ax.set_xlabel('Support'); ax.set_ylabel('Confidence')
plt.tight_layout()
plt.savefig('../report/images/nb05_ar_bubble.png', dpi=150, bbox_inches='tight')
plt.show()
print("Association Rules analysis complete!")